# Прогноз стоимости квартир в Москве

Учебный проект по регрессии: по характеристикам квартиры оценить её цену. Ноутбук проходит весь путь от исходного CSV до проверки модели и прогноза для новой квартиры. В каждом разделе объясняется, зачем нужен шаг.

> Важно: источник называет целевой столбец `real_price`, но до реального применения нужно отдельно проверить происхождение и смысл этих цен. Модель учится на данном наборе и сама по себе не гарантирует точную оценку рынка.

## 1. Формулируем задачу

Это задача **регрессии**: ответ модели — числовая величина. Целевой столбец — `real_price`; признаки — площадь, число комнат, ремонт и этажность. Мы оставим часть квартир в стороне до финальной оценки, чтобы проверить прогноз на примерах, которые модель не видела при обучении.

Основная метрика — **MAE**: средняя абсолютная ошибка в рублях. Также посмотрим на RMSE, которая сильнее учитывает крупные промахи, и R² — дополнительную оценку того, насколько прогнозы объясняют разброс цены.

## 2. Подключаем библиотеки

`pandas` читает и обрабатывает таблицы, `matplotlib` строит графики, а `scikit-learn` содержит инструменты разделения данных, подготовки признаков, моделей и метрик. В Google Colab эти библиотеки обычно уже доступны.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import RandomizedSearchCV, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 42
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")

## 3. Загружаем и изучаем исходную таблицу

Положи `main_data.csv` в ту же папку, откуда запущен ноутбук. Сначала читаем весь файл и проверяем его форму и города. Это помогает убедиться, что загрузился нужный набор данных.

In [ ]:
raw = pd.read_csv("main_data.csv")
print("Размер исходной таблицы:", raw.shape)
display(raw.head())
print("Количество квартир по городам:")
display(raw["city"].value_counts())

## 4. Оставляем Москву

В наборе есть несколько городов. Фильтруем строки по `city`, затем удаляем этот столбец: после фильтрации он одинаковый во всех строках и не помогает различать квартиры. `copy()` создаёт отдельную таблицу для дальнейших изменений.

In [ ]:
moscow = (
    raw.loc[raw["city"] == "Moscow"]
    .drop(columns="city")
    .reset_index(drop=True)
    .copy()
)
print("Размер московской выборки:", moscow.shape)
display(moscow.head())

## 5. Проверяем качество данных

Перед моделированием проверим пропуски, полные дубликаты и базовую логику значений. Мы не удаляем редкие дорогие квартиры автоматически: сначала нужно понять, ошибка ли это или настоящий объект. В данном наборе каждая строка должна описывать квартиру с положительными площадью, ценой и числом комнат, а этаж не должен превышать этажность дома.

In [ ]:
print("Типы и заполненность:")
moscow.info()
print("\nПропусков по столбцам:")
display(moscow.isna().sum())
print("Полных дубликатов:", moscow.duplicated().sum())

valid_rows = (
    (moscow["rooms"] > 0)
    & (moscow["m2"] > 0)
    & (moscow["real_price"] > 0)
    & (moscow["floor"] > 0)
    & (moscow["all_floor"] > 0)
    & (moscow["floor"] <= moscow["all_floor"])
)
print("Строк с корректными базовыми значениями:", valid_rows.sum(), "из", len(moscow))
moscow = moscow.loc[valid_rows].copy()

## 6. Изучаем распределения и связи

Сводная таблица показывает типичные значения и диапазоны. Медиана (`50%`) меньше зависит от редких дорогих квартир, чем среднее. Ниже отдельно смотрим цены по ремонту и связь цены с площадью. Цена за квадратный метр здесь только для анализа; в признаки модели её не включаем, так как она вычисляется из целевой цены.

In [ ]:
display(moscow.describe())
print("Медианная цена по типу ремонта:")
display(moscow.groupby("repair")["real_price"].median().sort_values(ascending=False))
print("Корреляция площади и цены:", moscow["m2"].corr(moscow["real_price"]))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
moscow.plot.scatter(x="m2", y="real_price", alpha=0.65, ax=axes[0])
axes[0].set_title("Цена и площадь")
axes[0].set_xlabel("Площадь, м²")
axes[0].set_ylabel("Цена, ₽")
moscow["real_price"].plot.hist(bins=30, ax=axes[1])
axes[1].set_title("Распределение цен")
axes[1].set_xlabel("Цена, ₽")
plt.tight_layout()
plt.show()

**Как читать графики:** на диаграмме рассеяния каждая точка — квартира. Общее движение точек вверх вправо означает, что более просторные квартиры обычно дороже. Разброс по вертикали показывает, что одной площади недостаточно. Гистограмма показывает, какие ценовые диапазоны встречаются чаще.

## 7. Создаём дополнительные признаки

Исходные признаки не полностью описывают положение квартиры в доме. Добавим долю этажа (`floor_ratio`), признаки первого и последнего этажа, площадь на комнату и признак высотного дома. Функция будет применяться одинаково к обучающим данным и к новой квартире.

In [ ]:
def make_features(data):
    result = data.copy()
    result["floor_ratio"] = result["floor"] / result["all_floor"]
    result["is_first"] = (result["floor"] == 1).astype(int)
    result["is_last"] = (result["floor"] == result["all_floor"]).astype(int)
    result["area_per_room"] = result["m2"] / result["rooms"]
    result["is_high_rise"] = (result["all_floor"] >= 25).astype(int)
    return result

featured = make_features(moscow)
display(featured.head())

## 8. Разделяем признаки, ответ и тестовую выборку

`X` содержит то, что модель может знать о квартире; `y` — цену, которую нужно предсказать. Сначала отделяем тестовую часть. Она не участвует ни в подготовке, ни в настройке моделей и используется только для итоговой проверки. `random_state` фиксирует случайное разбиение, чтобы результат можно было повторить.

In [ ]:
feature_columns = [
    "rooms", "m2", "repair", "floor", "all_floor",
    "floor_ratio", "is_first", "is_last", "area_per_room", "is_high_rise",
]
X = featured[feature_columns]
y = moscow["real_price"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE
)
print("Обучающая часть:", X_train.shape)
print("Тестовая часть:", X_test.shape)

## 9. Подготавливаем признаки внутри конвейера

Модели требуют числовой ввод, поэтому текстовый `repair` кодируем через one-hot encoding. Числовые признаки заполняем медианой и масштабируем для линейной модели. Эти преобразования объединены с моделью в `Pipeline`: параметры обработки вычисляются только на обучающей части и затем точно так же применяются к тесту и новым квартирам.

In [ ]:
numeric_features = [column for column in feature_columns if column != "repair"]
categorical_features = ["repair"]

numeric_preprocessor = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
categorical_preprocessor = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_preprocessor, numeric_features),
    ("categorical", categorical_preprocessor, categorical_features),
])

## 10. Сравниваем базовый и рабочие модели

Сначала создаём простой ориентир: `DummyRegressor` всегда предсказывает медианную цену обучающей выборки. Полезная модель должна заметно превзойти этот ориентир. Сравним Ridge (линейная модель с регуляризацией) и случайный лес. Пятиблочная кросс-валидация несколько раз проверяет модели на разных частях обучающего набора. Меньше MAE — лучше.

In [ ]:
models = {
    "Медианный ориентир": Pipeline([
        ("preprocess", preprocessor),
        ("model", DummyRegressor(strategy="median")),
    ]),
    "Ridge": Pipeline([
        ("preprocess", preprocessor),
        ("model", Ridge(alpha=10.0)),
    ]),
    "Random Forest": Pipeline([
        ("preprocess", preprocessor),
        ("model", RandomForestRegressor(
            n_estimators=300, min_samples_leaf=3, random_state=RANDOM_STATE, n_jobs=-1
        )),
    ]),
}

cv_results = []
for name, model in models.items():
    scores = -cross_val_score(
        model, X_train, y_train, cv=5, scoring="neg_mean_absolute_error", n_jobs=-1
    )
    cv_results.append({"Модель": name, "Средний MAE, ₽": scores.mean(), "Разброс MAE, ₽": scores.std()})

cv_table = pd.DataFrame(cv_results).sort_values("Средний MAE, ₽")
display(cv_table)

## 11. Настраиваем случайный лес

Выберем случайный лес и проверим несколько вариантов глубины дерева и минимального числа объектов в листе. Эти параметры ограничивают сложность модели и помогают сдерживать переобучение на небольшой выборке. Поиск использует только `X_train` и `y_train`; тестовая часть остаётся нетронутой. Если кросс-валидация показала, что другая модель лучше, этот раздел можно применить к ней вместо леса.

In [ ]:
forest_pipeline = models["Random Forest"]
param_distributions = {
    "model__max_depth": [None, 4, 6, 8, 12],
    "model__min_samples_leaf": [2, 3, 5, 8, 12],
    "model__max_features": [0.7, 0.9, 1.0],
}

search = RandomizedSearchCV(
    forest_pipeline,
    param_distributions=param_distributions,
    n_iter=15,
    scoring="neg_mean_absolute_error",
    cv=5,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
search.fit(X_train, y_train)
best_model = search.best_estimator_
print("Лучшие параметры:", search.best_params_)
print(f"Лучший MAE на обучающей кросс-валидации: {-search.best_score_:,.0f} ₽")

## 12. Один раз оцениваем модель на тесте

Теперь модель впервые получает `X_test`. Сравним её с медианным ориентиром по одним и тем же объектам. MAE читается как типичный размер ошибки в рублях; RMSE сильнее штрафует отдельные крупные ошибки. R² может быть отрицательным, если прогноз хуже простого предсказания среднего.

In [ ]:
y_pred = best_model.predict(X_test)
baseline = models["Медианный ориентир"].fit(X_train, y_train)
baseline_pred = baseline.predict(X_test)

def metrics_row(name, actual, predicted):
    return {
        "Модель": name,
        "MAE, ₽": mean_absolute_error(actual, predicted),
        "RMSE, ₽": np.sqrt(mean_squared_error(actual, predicted)),
        "R²": r2_score(actual, predicted),
    }

test_results = pd.DataFrame([
    metrics_row("Медианный ориентир", y_test, baseline_pred),
    metrics_row("Настроенный Random Forest", y_test, y_pred),
])
display(test_results)

## 13. Разбираем отдельные ошибки

Общая метрика скрывает конкретные примеры. Посмотрим на несколько квартир, где абсолютная ошибка оказалась наибольшей. Это помогает понять, какие объекты модель оценивает хуже всего.

In [ ]:
error_table = X_test.copy()
error_table["Цена, ₽"] = y_test
error_table["Прогноз, ₽"] = y_pred
error_table["Абсолютная ошибка, ₽"] = (error_table["Цена, ₽"] - error_table["Прогноз, ₽"]).abs()
display(error_table.sort_values("Абсолютная ошибка, ₽", ascending=False).head(10))

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(y_test, y_pred, alpha=0.7)
limits = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
ax.plot(limits, limits, linestyle="--", color="red")
ax.set_xlabel("Фактическая цена, ₽")
ax.set_ylabel("Прогноз, ₽")
ax.set_title("Фактическая цена и прогноз")
plt.show()

На графике идеальные прогнозы лежали бы на красной диагонали. Чем дальше точка от линии, тем больше ошибка. Ошибки могут быть связаны с отсутствующими в наборе характеристиками: районом, состоянием дома, близостью к метро и другими условиями.

## 14. Смотрим, какие признаки важны для модели

Permutation importance измеряет, насколько ухудшается качество, если перемешать значения одного признака в тестовой таблице. Чем больше рост MAE, тем больше модель опиралась на этот признак. Это оценка поведения модели, а не доказательство причин влияния на цену.

In [ ]:
importance = permutation_importance(
    best_model, X_test, y_test,
    scoring="neg_mean_absolute_error",
    n_repeats=20,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
importance_table = pd.DataFrame({
    "Признак": X_test.columns,
    "Рост MAE при перемешивании, ₽": importance.importances_mean,
}).sort_values("Рост MAE при перемешивании, ₽", ascending=False)
display(importance_table)

## 15. Делаем прогноз для новой квартиры

Прогнозировать нужно с теми же признаками, что использовались при обучении. Функция принимает исходные характеристики, создаёт производные признаки и передаёт таблицу в готовый конвейер. Пример ниже условный: квартира с двумя комнатами, площадью 60 м², на 5-м этаже 12-этажного дома, с ремонтом `needs`.

In [ ]:
def predict_apartment_price(rooms, m2, repair, floor, all_floor):
    apartment = pd.DataFrame([{
        "rooms": rooms,
        "m2": m2,
        "repair": repair,
        "floor": floor,
        "all_floor": all_floor,
    }])
    apartment_features = make_features(apartment)[feature_columns]
    return float(best_model.predict(apartment_features)[0])

example_price = predict_apartment_price(rooms=2, m2=60, repair="needs", floor=5, all_floor=12)
print(f"Прогноз цены: {example_price:,.0f} ₽")

## 16. Сохраняем готовый конвейер

В файл сохраняется вся цепочка: обработка пропусков, кодирование категорий, масштабирование и модель. Для нового запуска нужно загрузить эту цепочку и применять к входным данным ту же функцию создания признаков. В Colab файл находится во временной среде, поэтому для постоянного хранения его нужно скачать или сохранить на Google Drive.

In [ ]:
import joblib
joblib.dump(best_model, "moscow_apartment_price_model.joblib")
print("Модель сохранена в moscow_apartment_price_model.joblib")

## 17. Итог проекта

После выполнения ноутбука заполни выводы по фактическим результатам:

1. Сколько московских квартир осталось после проверки данных?
2. Какая модель победила медианный ориентир и на сколько?
3. Какова MAE на отложенной тестовой выборке в рублях?
4. Какие признаки модель использовала сильнее всего?
5. Какие данные стоит добавить, чтобы приблизить проект к реальной оценке квартир?

**Ограничения:** выборка содержит около 800 московских записей и не включает район, адрес, метро, возраст и тип дома. Качество прогноза ограничено как размером выборки, так и отсутствующими факторами. Результат — учебная модель на данном CSV, а не готовый сервис оценки недвижимости.